# Generate Synthetic Task Sets

This notebook converts the bundled A15 benchmark measurements into synthetic periodic
task sets. UUniFast controls total utilization, while execution times and deadlines are
sampled reproducibly. Generated files are written under `data/generated/`, which is
ignored by Git because it can be recreated from the benchmark inputs.


In [ ]:
from pathlib import Path
import csv
import random

import pandas as pd


def project_root() -> Path:
    """Locate the repository root when launched from either root or notebooks/."""
    current = Path.cwd().resolve()
    return current.parent if current.name == "notebooks" else current


ROOT = project_root()
BENCHMARK_DIR = ROOT / "data" / "benchmarks"
GENERATED_DIR = ROOT / "data" / "generated"
VOLTAGE = 1.1
FREQUENCY = 2.0e9


In [ ]:
def read_benchmarks(input_files, task_count, peak_profile, rng):
    """Sample task templates and derive the scheduler's power-model parameters."""
    templates = []
    measured_peaks = []

    for input_file in input_files:
        frame = pd.read_csv(input_file)
        static_power = frame["Static"].mean()
        dynamic_power = frame["Dynamic"].mean()
        measured_peaks.append(frame["Total"].max())

        templates.append({
            "name": input_file.stem,
            "alpha_C": dynamic_power / (VOLTAGE**2 * FREQUENCY),
            "I0_exp_term": static_power / VOLTAGE,
            "avg_power": frame["Average"].iloc[0],
            "static": static_power,
            "min": (frame["Total"].min() - static_power) / (VOLTAGE**2 * FREQUENCY),
        })

    if peak_profile not in {"normal", "high"}:
        raise ValueError("peak_profile must be 'normal' or 'high'")

    sampled = []
    lower_peak = 1.51 if peak_profile == "high" else min(measured_peaks)
    upper_peak = max(measured_peaks)
    for _ in range(task_count):
        task = rng.choice(templates).copy()
        peak_power = rng.uniform(lower_peak, upper_peak)
        task["peak"] = (peak_power - task["static"]) / (VOLTAGE**2 * FREQUENCY)
        sampled.append(task)
    return sampled


def uunifast(
    task_count,
    total_utilization,
    rng,
    min_task_utilization=0.001,
    max_task_utilization=0.4,
    attempts=1000,
):
    """Return bounded utilizations whose sum is total_utilization."""
    for _ in range(attempts):
        remaining = total_utilization
        values = []
        for index in range(task_count - 1):
            next_remaining = remaining * (rng.random() ** (1.0 / (task_count - index - 1)))
            values.append(remaining - next_remaining)
            remaining = next_remaining
        values.append(remaining)
        if all(min_task_utilization <= value <= max_task_utilization for value in values):
            return values
    raise RuntimeError("Unable to generate a bounded utilization vector")


def write_task_set(tasks, utilizations, output_dir, deadlines, rng):
    """Write one scheduler input CSV per synthetic task."""
    output_dir.mkdir(parents=True, exist_ok=True)
    for index, (task, utilization) in enumerate(zip(tasks, utilizations), start=1):
        valid_deadlines = [deadline for deadline in deadlines if deadline * utilization >= 5]
        if not valid_deadlines:
            raise ValueError(f"No valid deadline for task {index} at utilization {utilization:.4f}")
        deadline = rng.choice(valid_deadlines)
        execution_time = int(deadline * utilization)
        row = [
            execution_time,
            task["alpha_C"],
            task["I0_exp_term"],
            task["avg_power"],
            task["static"],
            task["min"],
            task["peak"],
            utilization,
            deadline,
        ]
        with (output_dir / f"task{index}.csv").open("w", newline="") as handle:
            writer = csv.writer(handle)
            writer.writerow(["#", "alpha_C", "I0_exp_term", "avg_power", "static", "min", "peak", "util", "deadline"])
            writer.writerow(row)


In [ ]:
SEED = 123
CORES = 4
TASKS_PER_CORE = 10
PEAK_PROFILE = "normal"
DEADLINES = [3200, 3000, 1500, 2000, 4000, 5000, 1600, 1250, 2500, 4800, 2400, 8000, 1200, 6000]

benchmark_files = sorted(BENCHMARK_DIR.glob("*.csv"))
if not benchmark_files:
    raise FileNotFoundError(f"No benchmark CSV files found in {BENCHMARK_DIR}")

for load_point in range(1, 11):
    rng = random.Random(SEED + load_point)
    task_count = CORES * TASKS_PER_CORE
    total_utilization = (load_point / 10) * CORES
    tasks = read_benchmarks(benchmark_files, task_count, PEAK_PROFILE, rng)
    utilizations = uunifast(task_count, total_utilization, rng)
    destination = GENERATED_DIR / f"tsp-{CORES}c" / f"limit_{total_utilization}" / "1"
    write_task_set(tasks, utilizations, destination, DEADLINES, rng)
    print(f"wrote {task_count} tasks to {destination.relative_to(ROOT)}")
